In [0]:
try:
    landing_table = dbutils.widgets.get("landing_table")
    history_table = dbutils.widgets.get("history_table")
except Exception as e:
    print(f"Error initializing variables: {e}")
    raise

In [0]:
try:
    spark.sql(f"""
        MERGE INTO {history_table} tgt
        USING (
            SELECT *
            FROM (
                SELECT *, ROW_NUMBER() OVER (PARTITION BY id ORDER BY _load_timestamp) AS rn
                FROM {landing_table}
            ) sub
            WHERE rn = 1
        ) src
        ON tgt.id = src.id
        AND tgt._modified_ts < src._load_timestamp
        AND tgt._active_flag = 1

        WHEN MATCHED THEN
            UPDATE SET
                tgt._active_flag = 0,
                tgt._modified_ts = src._load_timestamp
    """)
except Exception as e:
    print(f"Error updating {history_table}: {e}")
    raise

In [0]:
try:
    spark.sql(f"""
        MERGE INTO {history_table} tgt
        USING (
            SELECT *
            FROM (
                SELECT *, ROW_NUMBER() OVER (PARTITION BY id ORDER BY _load_timestamp) AS rn
                FROM {landing_table}
            ) sub
            WHERE rn = 1
        ) src
        ON tgt.id = src.id
        AND tgt._created_ts = src._load_timestamp

        WHEN NOT MATCHED
        THEN
            INSERT (
                _id,
                claim_type,
                patient_control_no,
                patient_ctl_no,
                claim_charge,
                patient_name,
                create_date,
                payer_name,
                payer_sequence,
                patient_dob,
                statement_start,
                from_date,
                statement_end,
                through_date,
                id,
                patient_id,
                trans_type,
                _file_name,
                _created_ts,
                _modified_ts,
                _active_flag
            )
            VALUES (
                src._id,
                src.claim_type,
                src.patient_control_no,
                src.patient_ctl_no,
                src.claim_charge,
                src.patient_name,
                src.create_date,
                src.payer_name,
                src.payer_sequence,
                src.patient_dob,
                src.statement_start,
                src.from_date,
                src.statement_end,
                src.through_date,
                src.id,
                src.patient_id,
                src.trans_type,
                src._file_name,
                src._load_timestamp,
                src._load_timestamp,
                1
            );
    """)
except Exception as e:
    print(f"Error inserting into {history_table}: {e}")
    raise